# 3D Gaussian Splatting — Colab 학습 노트북

> CUDA 11.8 설치 셀 실행 후 **반드시 런타임 재시작**이 필요합니다.
>
> COLMAP만 실행할 경우, 4번까지 실행한 뒤 바로 11번으로 넘어가면 됩니다.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. 3DGS 레포 클론
`--recursive`로 submodule까지 함께 클론합니다.

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 13.29 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects:

## 3. 빌드 도구 및 기본 패키지 설치

In [ ]:
!apt-get update -qq
!apt-get install -y build-essential cmake ninja-build git wget
!pip install -q plyfile

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
git is already the newest version (1:2.34.1-1ubuntu1.17).
wget is already the newest version (1.21.2-2ubuntu1.1).
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 84 not upgraded.
Need to get 111 kB of archives.
After this operation, 358 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ninja-build amd64 1.10.1-1 [111 kB]
Fetched 111 kB in 1s (97.5 kB/s)
Selecting previously unselected package ninja-build.
(Reading database ... 122354 files and directories currently installed

## 4. COLMAP 설치

In [ ]:
!apt-get install -y colmap

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libamd2 libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libcamd2 libccolamd2 libceres2
  libcholmod3 libcolamd2 libcxsparse3 libdouble-conversion3 libevdev2
  libfreeimage3 libgflags2.2 libglew2.2 libgoogle-glog0v5 libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libilmbase25 libinput-bin
  libinput10 libjxr0 libmd4c0 libmetis5 libmtdev1 libopenexr25 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libraw20
  librsvg2-common libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common
  libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0
  libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1 libxcomposite1
  libxkbcommon-x11-0 libxtst6 qt5-gtk-platformtheme qttranslations5-l10n
  session-migration
Suggested packages:
  gle

In [ ]:
# convert.py를 sequential_matching 옵션 지원하도록 패치
convert_path = "/content/gaussian-splatting/convert.py"

with open(convert_path, "r") as f:
    code = f.read()

# 인자 추가
code = code.replace(
    'parser.add_argument("--skip_matching", action=\'store_true\')',
    'parser.add_argument("--skip_matching", action=\'store_true\')\nparser.add_argument("--sequential_matching", action=\'store_true\')'
)

# exhaustive_matcher → sequential_matcher 조건부 분기
old_matching = '''    ## Feature matching
    feat_matching_cmd = colmap_command + \" exhaustive_matcher \\
        --database_path \" + args.source_path + \"/distorted/database.db \\
        --SiftMatching.use_gpu \" + str(use_gpu)'''

new_matching = '''    ## Feature matching
    if args.sequential_matching:
        feat_matching_cmd = colmap_command + \" sequential_matcher \\
        --database_path \" + args.source_path + \"/distorted/database.db \\
        --SiftMatching.use_gpu \" + str(use_gpu)
    else:
        feat_matching_cmd = colmap_command + \" exhaustive_matcher \\
        --database_path \" + args.source_path + \"/distorted/database.db \\
        --SiftMatching.use_gpu \" + str(use_gpu)'''

code = code.replace(old_matching, new_matching)

with open(convert_path, "w") as f:
    f.write(code)

print("패치 완료")

패치 완료


## 5. CUDA 11.8 설치
> ⚠️ **이 셀 실행 완료 후 런타임을 재시작하세요.**
> 메뉴: 런타임 → 런타임 다시 시작
> 재시작 후 **6번 셀(환경변수 설정)부터** 실행하세요.

In [ ]:
import subprocess, os
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

if 'release 12' in result.stdout or 'release 11' not in result.stdout:
    print('Installing CUDA 11.8...')
    os.system('wget -q https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run')
    os.system('sh cuda_11.8.0_520.61.05_linux.run --silent --toolkit')
    print('Done. ⚠️  Runtime restart required!')
else:
    print('CUDA 11.x already present, skipping.')

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

Installing CUDA 11.8...
Done. ⚠️  Runtime restart required!


## 6. CUDA 11.8 환경 변수 설정
**런타임 재시작 후 이 셀부터 실행합니다.**

In [ ]:
import os, datetime

os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] = f"/usr/local/cuda-11.8/bin:{os.environ['PATH']}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-11.8/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


## 7. PyTorch 재설치 (CUDA 11.8 호환)

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 \
    --index-url https://download.pytorch.org/whl/cu118
!pip install numpy==1.26.4

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Files removed: 6
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 130.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 109.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 71.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 126.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━

## 8. torch.py 충돌 확인

In [ ]:
import os
if os.path.exists('./torch.py'):
    os.rename('torch.py', 'torch_local.py')
    print('Renamed torch.py -> torch_local.py')
else:
    print('✅ No conflict.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

✅ No conflict.
PyTorch : 2.2.2+cu118
CUDA    : True
GPU     : NVIDIA L4


## 9. Submodule 빌드 및 설치
`diff-gaussian-rasterization`과 `simple-knn`을 빌드합니다. (각 5~10분 소요)

In [ ]:
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!python setup.py install
# submodule 재빌드
import subprocess

cmds = [
    "pip uninstall diff_gaussian_rasterization -y",
    "cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization && pip install -e .",
]

for cmd in cmds:
    result = subprocess.run(cmd, shell=True, capture_output=False)
    print("✅" if result.returncode == 0 else f"❌ (exit {result.returncode})")

/content/gaussian-splatting/submodules/diff-gaussian-rasterization
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/i

In [ ]:
%cd /content/gaussian-splatting/submodules/simple-knn
!python setup.py install

/content/gaussian-splatting/submodules/simple-knn
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other

## 10. 설치 확인

In [ ]:
try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print('✅ diff_gaussian_rasterization OK')
except ImportError as e:
    print(f'❌ {e}')

try:
    import simple_knn
    print('✅ simple_knn OK')
except ImportError as e:
    print(f'❌ {e}')

✅ diff_gaussian_rasterization OK
✅ simple_knn OK


## 11. 전처리 → 학습 → 렌더링 → 평가 (Convert → Train → Render → Metrics)

- `convert.py`: COLMAP으로 SfM 수행 → `sparse/`, `images/` 생성
- `train.py`: Gaussian Splatting 학습
- `render.py` / `metrics.py`: 결과 렌더링 및 평가

```
OUTPUT_BASE/{scene}/
├── input/          ← raw 이미지
├── images/         ← undistorted (convert.py 결과)
├── sparse/         ← COLMAP sparse reconstruction
└── point_cloud/
    ├── iteration_10000/point_cloud.ply
    ├── iteration_20000/point_cloud.ply
    └── iteration_30000/point_cloud.ply
```

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
DATA_BASE = '/content/drive/MyDrive/Research/26_Capstone2/input'
OUTPUT_BASE = '/content/drive/MyDrive/Research/26_Capstone2/3DGS'

# ✏️ 처리할 scene 목록
# 'grass_rain',  'grass_snow',
# 'hydrant_rain','hydrant_snow',
# 'pillar_rain', 'pillar_snow',
# 'road_rain',   'road_snow',
# 'sky_rain',    'sky_snow',
# 'stair_rain',  'stair_snow',

# ✏️ 파라미터
ITERATIONS      = 30000
SAVE_ITERATIONS = '10000 20000 30000'

In [ ]:
import subprocess
result = subprocess.run(
    ["find", DATA_BASE, "-name", ".DS_Store", "-delete"],
    capture_output=False, text=True
)
print("삭제 완료")

삭제 완료


In [ ]:
# 1. convert
import os, textwrap, subprocess
from pathlib import Path

SCENES = [
    'pillar_original',
    'pillar_snow'
]

completed_convert = []
failed_convert    = []

for scene in SCENES:
    input_path  = f'{DATA_BASE}/{scene}'

    script = textwrap.dedent(f"""
        #!/bin/bash
        set -e
        ulimit -n 4096
        cd /content/gaussian-splatting

        echo '===== COLMAP 전처리 (convert.py): {scene} ====='
        QT_QPA_PLATFORM=offscreen python convert.py -s {input_path} --no_gpu --sequential_matching
    """).strip()

    script_path = Path('/content/gaussian-splatting/scripts/run_convert.sh')
    script_path.parent.mkdir(exist_ok=True, parents=True)
    script_path.write_text(script, encoding='utf-8')

    print(f"\n{'='*60}\n[Convert] Scene: {scene}\n{'='*60}")

    with subprocess.Popen(
        ['bash', str(script_path)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    ) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)

    if proc.returncode != 0:
        print(f'\n❌ [{scene}] Convert 실패 (exit {proc.returncode})')
        failed_convert.append(scene)
    else:
        print(f'\n✅ [{scene}] Convert 완료')
        completed_convert.append(scene)

print(f'\n{"="*60}')
print(f'Convert 완료: {completed_convert}')
print(f'Convert 실패: {failed_convert}')

In [ ]:
# 2. train + render + metrics
import random, textwrap, subprocess, datetime, os
from pathlib import Path
from datetime import timezone, timedelta
import torch, gc

KST = timezone(timedelta(hours=9))

SCENES = [
    'grass_original',
    'grass_snow',
    'hydrant_original',
    'hydrant_snow'
]

completed = []
failed    = []

scenes_to_run = [s for s in SCENES if s not in failed_convert]

for scene in scenes_to_run:
    torch.cuda.empty_cache()
    gc.collect()

    timestamp   = datetime.datetime.now(KST).strftime('%Y-%m-%d_%H:%M')
    input_path  = f'{DATA_BASE}/{scene}'
    output_path = f'{OUTPUT_BASE}/{scene}/{timestamp}'
    port        = random.randint(10000, 30000)

    os.makedirs(output_path, exist_ok=True)

    if 'output_paths' not in dir():
        output_paths = {}
    output_paths[scene] = output_path

    script = textwrap.dedent(f"""
        #!/bin/bash
        set -e
        ulimit -n 4096
        cd /content/gaussian-splatting

        echo '===== Training: {scene} ====='
        python train.py \\
            -s {input_path} \\
            -m {output_path} \\
            --ip 127.0.0.1 --port {port} \\
            --eval \\
            --iterations {ITERATIONS} \\
            --save_iterations {SAVE_ITERATIONS}

        echo '===== Rendering: {scene} ====='
        python render.py -m {output_path}

        echo '===== Metrics: {scene} ====='
        python metrics.py -m {output_path}
    """).strip()

    script_path = Path('/content/gaussian-splatting/scripts/run_train_eval.sh')
    script_path.parent.mkdir(exist_ok=True, parents=True)
    script_path.write_text(script, encoding='utf-8')

    print(f"\n{'='*60}\n[Train+Eval] Scene: {scene}\n{'='*60}")

    with subprocess.Popen(
        ['bash', str(script_path)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    ) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)

    if proc.returncode != 0:
        print(f'\n❌ [{scene}] 실패 (exit {proc.returncode})')
        failed.append(scene)
    else:
        print(f'\n✅ [{scene}] 완료')
        completed.append(scene)

print(f'\n{"="*60}')
print(f'완료: {completed}')
print(f'실패: {failed}')

Streaming output truncated to the last 5000 lines.
Training progress:  67%|██████▋   | 20000/30000 [14:06<07:38, 21.83it/s, Loss=0.0053822, Depth Loss=0.0000000]
[ITER 20000] Saving Gaussians [15/04 18:47:11]

Training progress: 100%|██████████| 30000/30000 [21:50<00:00, 22.89it/s, Loss=0.0050401, Depth Loss=0.0000000]

[ITER 30000] Evaluating test: L1 0.06643257290124893 PSNR 18.796512603759766 [15/04 18:54:57]

[ITER 30000] Evaluating train: L1 0.004522408312186599 PSNR 43.64293212890625 [15/04 18:54:59]

[ITER 30000] Saving Gaussians [15/04 18:54:59]

Training complete. [15/04 18:55:07]
===== Rendering: hydrant_original =====
Looking for config file in /content/drive/MyDrive/Research/26_Capstone2/3DGS/hydrant_original/2026-04-16_03:32/cfg_args
Config file found: /content/drive/MyDrive/Research/26_Capstone2/3DGS/hydrant_original/2026-04-16_03:32/cfg_args
Rendering /content/drive/MyDrive/Research/26_Capstone2/3DGS/hydrant_original/2026-04-16_03:32
Loading trained model at iteration 30